# Spectrogram Models — CNN & CRNN

Track-level emotion classification from Mel-spectrograms.

**Pipeline:** audio → Mel-spectrogram → `emotion_quadrant`  
**Splits:** existing track-level `song_id` splits (no random frame splitting)

In [1]:
from pathlib import Path
import logging
import sys

import pandas as pd
import torch

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_project_root, load_configs, resolve_path
from src.features.mel_spectrograms import extract_mel_spectrograms_dataset, load_mel_spectrogram_index
from src.training.train_sequence import log_training_device, resolve_device
from src.training.train_spectrogram import train_and_evaluate_spectrogram_models

configs = load_configs(PROJECT_ROOT)
root = get_project_root(PROJECT_ROOT)

if torch.cuda.is_available():
    configs["training"]["general"]["device"] = "cuda"
    print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)}")
else:
    print(
        "WARNING: CUDA not available — spectrogram models will train on CPU. "
        "Install a CUDA-enabled PyTorch build for your RTX 4060 Ti."
    )

device = resolve_device(str(configs["training"]["general"]["device"]))
log_training_device(device)

12:42:35 | INFO | src.training.train_sequence | Training device: cuda (NVIDIA GeForce RTX 4060 Ti)


PyTorch 2.11.0+cu128 | CUDA 12.8 | NVIDIA GeForce RTX 4060 Ti


## 1. Extract or load Mel-spectrograms

In [2]:
index_path = resolve_path(root, configs["paths"]["features"]["mel_spectrogram_index"])

if index_path.exists():
    index_df = load_mel_spectrogram_index(configs)
    print(f"Loaded index: {index_path} ({len(index_df)} tracks)")
else:
    from src.data.load_deam import build_metadata_table
    metadata = build_metadata_table(configs)
    index_df = extract_mel_spectrograms_dataset(metadata, configs)
    print(f"Extracted Mel-spectrograms for {len(index_df)} tracks")

index_df.head()

Loaded index: C:\Users\athen\Desktop\music-emotion-recognition\data\features\mel_spectrogram_index.csv (1802 tracks)


,song_id,spectrogram_path,split,emotion_quadrant,n_mels,n_frames
0,2,mel_spectrograms/2.npy,train,Q4,128,1938
1,3,mel_spectrograms/3.npy,train,Q4,128,1938
2,4,mel_spectrograms/4.npy,train,Q1,128,1938
3,5,mel_spectrograms/5.npy,val,Q3,128,1938
4,7,mel_spectrograms/7.npy,val,Q1,128,1938


## 2. Train CNN and CRNN (run when ready — can take a long time)

In [3]:
summary_df = train_and_evaluate_spectrogram_models(
    configs,
    model_names=("cnn", "crnn"),
    extract_features=False,
)
summary_df

12:43:26 | INFO | src.training.train_sequence | Training device: cuda (NVIDIA GeForce RTX 4060 Ti)
12:43:26 | INFO | src.training.train_spectrogram | Spectrogram split validation passed (track-level).


Training spectrogram models:   0%|          | 0/2 [00:00<?, ?it/s]

12:43:26 | INFO | src.training.train_spectrogram | Training spectrogram model: cnn on cuda
12:43:41 | INFO | src.training.train_spectrogram | cnn epoch 1/40 — train_loss=1.2890, val_macro_f1=0.2969
12:43:52 | INFO | src.training.train_spectrogram | cnn epoch 2/40 — train_loss=1.2313, val_macro_f1=0.3170
12:44:04 | INFO | src.training.train_spectrogram | cnn epoch 3/40 — train_loss=1.1876, val_macro_f1=0.2655
12:44:15 | INFO | src.training.train_spectrogram | cnn epoch 4/40 — train_loss=1.1695, val_macro_f1=0.3122
12:44:27 | INFO | src.training.train_spectrogram | cnn epoch 5/40 — train_loss=1.1464, val_macro_f1=0.3221
12:44:40 | INFO | src.training.train_spectrogram | cnn epoch 6/40 — train_loss=1.1326, val_macro_f1=0.3179
12:44:52 | INFO | src.training.train_spectrogram | cnn epoch 7/40 — train_loss=1.1517, val_macro_f1=0.3177
12:45:05 | INFO | src.training.train_spectrogram | cnn epoch 8/40 — train_loss=1.1157, val_macro_f1=0.3201
12:45:17 | INFO | src.training.train_spectrogram | cn

,model_name,task_type,feature_type,target_type,eval_split,train_size,val_size,test_size,accuracy,macro_f1,weighted_f1,best_val_macro_f1
0,cnn,spectrogram,mel_spectrogram,emotion_quadrant,val,1261,270,271,0.603704,0.348356,0.510954,0.348356
1,cnn,spectrogram,mel_spectrogram,emotion_quadrant,test,1261,270,271,0.608856,0.351866,0.514189,0.348356
2,crnn,spectrogram,mel_spectrogram,emotion_quadrant,val,1261,270,271,0.637037,0.367961,0.539469,0.367961
3,crnn,spectrogram,mel_spectrogram,emotion_quadrant,test,1261,270,271,0.638376,0.370703,0.540945,0.367961


In [4]:
test_df = summary_df[summary_df["eval_split"] == "test"]
test_df[["model_name", "accuracy", "macro_f1", "weighted_f1"]]

,model_name,accuracy,macro_f1,weighted_f1
1,cnn,0.608856,0.351866,0.514189
3,crnn,0.638376,0.370703,0.540945
